# 01 — Dataset exploration

This notebook inspects the **local dataset as it exists**. It does not assume the documented image count, class names, or signer metadata are correct. Configure `data.dataset_root` in `configs/config.yaml` to point to a directory whose immediate child directories are the candidate class folders, then run the notebook from top to bottom. Images are read-only; this notebook does not rename, move, or delete dataset files.

If the dataset path is not configured or does not exist, the notebook displays setup guidance and skips data-dependent cells without fabricating a report.

In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
import hashlib
import json

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from PIL import Image, ImageStat, UnidentifiedImageError

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"

if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Project configuration was not found: {CONFIG_PATH}")

with CONFIG_PATH.open("r", encoding="utf-8") as config_file:
    CONFIG = yaml.safe_load(config_file) or {}

configured_root = CONFIG.get("data", {}).get("dataset_root")
if configured_root:
    candidate = Path(configured_root).expanduser()
    DATASET_ROOT = (PROJECT_ROOT / candidate).resolve() if not candidate.is_absolute() else candidate.resolve()
else:
    DATASET_ROOT = None

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
DATASET_READY = DATASET_ROOT is not None and DATASET_ROOT.is_dir()

print(f"Project root: {PROJECT_ROOT}")
print(f"Configured dataset path: {DATASET_ROOT if DATASET_ROOT else 'not configured'}")
if DATASET_ROOT is None:
    print("Set data.dataset_root in configs/config.yaml to the local dataset directory, then rerun this notebook.")
elif not DATASET_ROOT.is_dir():
    print(f"Dataset directory does not exist: {DATASET_ROOT}")
    print("Correct data.dataset_root in configs/config.yaml. No dataset results were produced.")
else:
    print("Dataset directory found. Continue with the inspection cells below.")

## Folder structure and candidate labels

The first directory under the configured root is treated as a candidate class folder for inventory only. Review this assumption against the observed structure; the notebook does not rename labels or assert that folder names are correct ISL word labels.

In [ ]:
if DATASET_READY:
    top_level = sorted(DATASET_ROOT.iterdir(), key=lambda path: path.name.casefold())
    print("Top-level entries:")
    for path in top_level:
        kind = "directory" if path.is_dir() else "file"
        print(f"  [{kind}] {path.name}")

    class_directories = [path for path in top_level if path.is_dir()]
    image_records = []
    for class_dir in class_directories:
        for image_path in sorted(class_dir.rglob("*")):
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
                image_records.append({
                    "path": image_path,
                    "relative_path": image_path.relative_to(DATASET_ROOT).as_posix(),
                    "label": class_dir.name,
                    "extension": image_path.suffix.lower(),
                })
    images_df = pd.DataFrame(image_records, columns=["path", "relative_path", "label", "extension"])
    print(f"\nCandidate class directories: {len(class_directories)}")
    print(f"Recognized image files: {len(images_df)}")
    print("Candidate labels:", [path.name for path in class_directories])
else:
    class_directories = []
    image_records = []
    images_df = pd.DataFrame(columns=["path", "relative_path", "label", "extension"])
    print("Skipped: configure a valid local dataset directory first.")

In [ ]:
if DATASET_READY:
    all_entries = sorted(DATASET_ROOT.rglob("*"))
    print(f"Recursive entries under dataset root: {len(all_entries)}")
    for entry in all_entries[:100]:
        relative = entry.relative_to(DATASET_ROOT)
        marker = "/" if entry.is_dir() else ""
        print(f"  {relative.as_posix()}{marker}")
    if len(all_entries) > 100:
        print(f"  ... {len(all_entries) - 100} more entries omitted")
else:
    print("Skipped: dataset path is unavailable.")

## Image counts and file formats

Counts are based on recognized image extensions and candidate labels derived from the first directory level. Empty class folders are retained in the count table so potential class imbalance is visible.

In [ ]:
if DATASET_READY:
    counts = Counter(record["label"] for record in image_records)
    class_counts = pd.DataFrame([
        {"candidate_label": folder.name, "image_count": counts[folder.name]}
        for folder in class_directories
    ], columns=["candidate_label", "image_count"])
    if not class_counts.empty:
        class_counts = class_counts.sort_values(["image_count", "candidate_label"], ascending=[False, True])
    display(class_counts)
    if not class_counts.empty:
        ax = class_counts.sort_values("candidate_label").plot.bar(
            x="candidate_label", y="image_count", legend=False, figsize=(max(10, len(class_counts) * 0.55), 5)
        )
        ax.set_title("Image count by candidate class folder")
        ax.set_xlabel("Candidate folder label")
        ax.set_ylabel("Image files")
        plt.xticks(rotation=60, ha="right")
        plt.tight_layout()
        plt.show()
    else:
        print("No immediate class directories were found beneath the configured dataset root.")
    print("\nImage files by extension:")
    display(images_df["extension"].value_counts().rename_axis("extension").to_frame("count"))
else:
    print("Skipped: no dataset inventory is available.")

## Readability, dimensions, and exact duplicates

Each recognized image is opened and verified without modifying it. Exact duplicates are identified by SHA-256 file content; this does not identify visually similar files encoded differently. Dimension and format summaries include only images that PIL can open.

In [ ]:
if DATASET_READY:
    image_details = []
    unreadable = []
    hashes = defaultdict(list)
    for record in image_records:
        path = record["path"]
        try:
            with Image.open(path) as image:
                image.verify()
            with Image.open(path) as image:
                gray_stats = ImageStat.Stat(image.convert("L"))
                image_details.append({
                    **{key: record[key] for key in ("relative_path", "label", "extension")},
                    "width": image.width,
                    "height": image.height,
                    "mode": image.mode,
                    "format": image.format,
                    "brightness_mean": gray_stats.mean[0],
                    "contrast_std": gray_stats.stddev[0],
                })
            digest_builder = hashlib.sha256()
            with path.open("rb") as image_file:
                for chunk in iter(lambda: image_file.read(1024 * 1024), b""):
                    digest_builder.update(chunk)
            digest = digest_builder.hexdigest()
            hashes[digest].append(record["relative_path"])
        except (OSError, UnidentifiedImageError, ValueError) as error:
            unreadable.append({"relative_path": record["relative_path"], "error": str(error)})

    details_df = pd.DataFrame(image_details)
    duplicate_groups = [paths for paths in hashes.values() if len(paths) > 1]
    print(f"Readable images: {len(details_df)} / {len(image_records)}")
    print(f"Unreadable/corrupted image files: {len(unreadable)}")
    print(f"Exact duplicate groups (SHA-256): {len(duplicate_groups)}")
    if unreadable:
        display(pd.DataFrame(unreadable))
    if duplicate_groups:
        display(pd.DataFrame({"duplicate_paths": [" | ".join(group) for group in duplicate_groups]}))
    if not details_df.empty:
        print("\nImage dimensions (width x height):")
        display((details_df.assign(dimensions=details_df["width"].astype(str) + " x " + details_df["height"].astype(str))
                 .groupby("dimensions").size().sort_values(ascending=False).rename("count").to_frame()))
        print("\nImage color modes:")
        display(details_df["mode"].value_counts().rename_axis("mode").to_frame("count"))
        print("\nPer-image brightness/contrast summary (grayscale mean and standard deviation):")
        display(details_df[["brightness_mean", "contrast_std"]].describe().round(2))
else:
    image_details = []
    unreadable = []
    duplicate_groups = []
    details_df = pd.DataFrame()
    print("Skipped: no dataset inventory is available.")

## Near-duplicate candidates

A 64-bit average hash is computed on readable images. Candidate pairs are found by splitting hashes into eight 8-bit chunks and comparing pairs sharing at least one chunk, then retaining Hamming distances of at most 5 and mean-RGB distances of at most 60 (0–255 scale). This remains a screening heuristic, not proof of duplicate content; visually inspect candidates before deciding whether any should be excluded from a split.

In [ ]:
def average_hash(image: Image.Image, hash_size: int = 8) -> int:
    """Return a simple integer average hash for an image."""
    gray = image.convert("L").resize((hash_size, hash_size), Image.Resampling.LANCZOS)
    pixels = list(gray.getdata())
    average = sum(pixels) / len(pixels)
    value = 0
    for pixel in pixels:
        value = (value << 1) | int(pixel >= average)
    return value


def near_duplicate_candidates(records, max_distance: int = 5, max_candidate_pairs: int = 200_000):
    """Return close-hash pairs, with a bound on candidate-pair memory and work."""
    hashed = []
    failures = []
    for record in records:
        try:
            with Image.open(record["path"]) as image:
                mean_rgb = tuple(ImageStat.Stat(image.convert("RGB").resize((16, 16))).mean)
                hashed.append((record["relative_path"], average_hash(image), mean_rgb))
        except (OSError, UnidentifiedImageError, ValueError) as error:
            failures.append((record["relative_path"], str(error)))

    buckets = defaultdict(set)
    for index, item in enumerate(hashed):
        value = item[1]
        for chunk_index in range(8):
            chunk = (value >> (chunk_index * 8)) & 0xFF
            buckets[(chunk_index, chunk)].add(index)

    candidate_pairs = set()
    candidate_limit_reached = False
    for indices in buckets.values():
        ordered = sorted(indices)
        for offset, first in enumerate(ordered):
            for second in ordered[offset + 1:]:
                candidate_pairs.add((first, second))
                if len(candidate_pairs) >= max_candidate_pairs:
                    candidate_limit_reached = True
                    break
            if candidate_limit_reached:
                break
        if candidate_limit_reached:
            break

    matches = []
    for first, second in candidate_pairs:
        distance = (hashed[first][1] ^ hashed[second][1]).bit_count()
        color_distance = sum((hashed[first][2][channel] - hashed[second][2][channel]) ** 2 for channel in range(3)) ** 0.5
        if distance <= max_distance and color_distance <= 60:
            matches.append({
                "image_a": hashed[first][0],
                "image_b": hashed[second][0],
                "hamming_distance": distance,
                "mean_rgb_distance": color_distance,
            })
    matches_df = pd.DataFrame(matches, columns=["image_a", "image_b", "hamming_distance", "mean_rgb_distance"])
    if not matches_df.empty:
        matches_df = matches_df.sort_values("hamming_distance").reset_index(drop=True)
    return matches_df, failures, candidate_limit_reached


if DATASET_READY:
    near_duplicates_df, hash_failures, candidate_limit_reached = near_duplicate_candidates(image_records)
    print(f"Near-duplicate candidate pairs (Hamming distance <= 5; mean-RGB distance <= 60): {len(near_duplicates_df)}")
    if candidate_limit_reached:
        print("Candidate-pair search reached the safety limit; near-duplicate results are incomplete.")
    if not near_duplicates_df.empty:
        display(near_duplicates_df.head(200))
        if len(near_duplicates_df) > 200:
            print("Showing the first 200 candidate pairs.")
    if hash_failures:
        print(f"Files that could not be hashed: {len(hash_failures)}")
else:
    near_duplicates_df = pd.DataFrame(columns=["image_a", "image_b", "hamming_distance", "mean_rgb_distance"])
    hash_failures = []
    candidate_limit_reached = False
    print("Skipped: no dataset inventory is available.")

## Sample images and metadata candidates

The first readable images from each candidate class are displayed for visual inspection. Metadata files and signer-related path/name tokens are reported as candidates only; their contents and meaning must be verified before using them for a signer-independent split.

In [ ]:
if DATASET_READY:
    readable_paths = {detail["relative_path"] for detail in image_details}
    sample_records = [record for record in image_records if record["relative_path"] in readable_paths]
    grouped_samples = defaultdict(list)
    for record in sample_records:
        if len(grouped_samples[record["label"]]) < 3:
            grouped_samples[record["label"]].append(record)

    if grouped_samples:
        columns = min(3, max(len(rows) for rows in grouped_samples.values()))
        rows = len(grouped_samples)
        fig, axes = plt.subplots(rows, columns, figsize=(4 * columns, 3.5 * rows), squeeze=False)
        for row_index, label in enumerate(sorted(grouped_samples)):
            for column_index in range(columns):
                axis = axes[row_index][column_index]
                if column_index < len(grouped_samples[label]):
                    record = grouped_samples[label][column_index]
                    with Image.open(record["path"]) as image:
                        axis.imshow(image.convert("RGB"))
                    axis.set_title(f"{label}\n{record['relative_path']}", fontsize=8)
                else:
                    axis.axis("off")
                axis.axis("off")
        plt.tight_layout()
        plt.show()
    else:
        print("No readable sample images were found.")

    signer_tokens = ("signer", "participant", "subject", "person", "volunteer")
    metadata_extensions = {".csv", ".tsv", ".json", ".jsonl", ".xlsx", ".xls", ".txt"}
    metadata_files = [path for path in DATASET_ROOT.rglob("*") if path.is_file() and path.suffix.lower() in metadata_extensions]
    print("Metadata-file candidates:")
    if metadata_files:
        for path in metadata_files:
            print(f"  {path.relative_to(DATASET_ROOT).as_posix()}")
        tabular_metadata = []
        for path in metadata_files:
            if path.suffix.lower() in {".csv", ".tsv"}:
                try:
                    separator = "\t" if path.suffix.lower() == ".tsv" else ","
                    columns = pd.read_csv(path, sep=separator, nrows=0).columns.astype(str).tolist()
                    signer_columns = [column for column in columns if any(token in column.casefold() for token in signer_tokens)]
                    tabular_metadata.append({
                        "file": path.relative_to(DATASET_ROOT).as_posix(),
                        "columns": columns,
                        "signer_like_columns": signer_columns,
                    })
                except (OSError, UnicodeError, pd.errors.ParserError, pd.errors.EmptyDataError) as error:
                    tabular_metadata.append({"file": path.relative_to(DATASET_ROOT).as_posix(), "read_error": str(error)})
        if tabular_metadata:
            print("Tabular metadata headers (signer-like fields are candidates, not verified identities):")
            display(pd.DataFrame(tabular_metadata))
    else:
        print("  None found by extension scan.")

    signer_path_candidates = [
        path.relative_to(DATASET_ROOT).as_posix()
        for path in DATASET_ROOT.rglob("*")
        if any(token in path.name.casefold() for token in signer_tokens)
    ]
    print("\nPaths with signer/participant-like names (candidates, not verified metadata):")
    if signer_path_candidates:
        for candidate in signer_path_candidates[:100]:
            print(f"  {candidate}")
        if len(signer_path_candidates) > 100:
            print(f"  ... {len(signer_path_candidates) - 100} more omitted")
    else:
        print("  None found by name scan. This does not prove signer metadata is absent.")
else:
    metadata_files = []
    signer_path_candidates = []
    print("Skipped: dataset path is unavailable.")

## Dataset summary

This summary contains only observations computed from the configured local directory. Dataset label correctness, signer identity, and near-duplicate overlap require manual/metadata validation before preprocessing or splitting.

In [ ]:
if DATASET_READY:
    observed_classes = len(class_directories)
    total_images = len(image_records)
    if not class_counts.empty and class_counts["image_count"].sum() > 0:
        nonzero_counts = class_counts.loc[class_counts["image_count"] > 0, "image_count"]
        imbalance_ratio = float(nonzero_counts.max() / nonzero_counts.min())
    else:
        imbalance_ratio = None

    summary = {
        "dataset_root": str(DATASET_ROOT),
        "candidate_class_directories": observed_classes,
        "expected_class_count_from_config": CONFIG.get("data", {}).get("expected_num_classes"),
        "recognized_image_files": total_images,
        "readable_images": len(image_details),
        "unreadable_images": len(unreadable),
        "exact_duplicate_groups": len(duplicate_groups),
        "near_duplicate_candidate_pairs": len(near_duplicates_df),
        "metadata_file_candidates": len(metadata_files),
        "signer_path_name_candidates": len(signer_path_candidates),
        "max_to_min_nonempty_class_count_ratio": imbalance_ratio,
        "labels_verified_as_intended_isl_words": False,
        "signer_metadata_verified": False,
    }
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    report_dir = PROJECT_ROOT / CONFIG.get("paths", {}).get("logs", "outputs/logs")
    report_dir.mkdir(parents=True, exist_ok=True)
    (report_dir / "dataset_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
    class_counts.to_csv(report_dir / "dataset_class_counts.csv", index=False)
    details_df.to_csv(report_dir / "dataset_image_details.csv", index=False)
    pd.DataFrame(unreadable, columns=["relative_path", "error"]).to_csv(report_dir / "dataset_unreadable_images.csv", index=False)
    pd.DataFrame({"duplicate_paths": [" | ".join(group) for group in duplicate_groups]}).to_csv(report_dir / "dataset_exact_duplicate_groups.csv", index=False)
    near_duplicates_df.to_csv(report_dir / "dataset_near_duplicate_candidates.csv", index=False)
    print(f"Dataset inspection reports saved under: {report_dir}")
else:
    print("No dataset summary produced. Configure a valid data.dataset_root and rerun the notebook.")